# Module 3 - Simulating Polarisation Dynamics

**Time:** 13:45-15:15  
**Tool focus:** NDlib plus explicit bounded-confidence simulation code

The notebook keeps the simulation logic visible so participants can see exactly how confidence thresholds and biased exposure alter the opinion trajectories.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / ".cache"
(CACHE_DIR / "matplotlib").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

import matplotlib.pyplot as plt
import ndlib
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(context="talk", style="whitegrid")
print("NDlib version:", getattr(ndlib, "__version__", "installed"))


In [ ]:
def build_demo_graph(seed=42):
    rng = np.random.default_rng(seed)
    communities = [
        {"label": "left-mainstream", "camp": "left", "size": 36, "mean_opinion": -0.55, "enclave": 0},
        {"label": "left-enclave", "camp": "left", "size": 18, "mean_opinion": -0.88, "enclave": 1},
        {"label": "right-mainstream", "camp": "right", "size": 36, "mean_opinion": 0.55, "enclave": 0},
        {"label": "right-enclave", "camp": "right", "size": 18, "mean_opinion": 0.88, "enclave": 1},
    ]
    sizes = [community["size"] for community in communities]
    probability_matrix = [
        [0.18, 0.08, 0.02, 0.004],
        [0.08, 0.25, 0.006, 0.001],
        [0.02, 0.006, 0.18, 0.08],
        [0.004, 0.001, 0.08, 0.25],
    ]

    graph = nx.stochastic_block_model(sizes, probability_matrix, seed=seed)
    graph.graph.clear()

    node_id = 0
    for community in communities:
        for _ in range(community["size"]):
            graph.nodes[node_id]["label"] = f"user_{node_id:03d}"
            graph.nodes[node_id]["community_label"] = community["label"]
            graph.nodes[node_id]["camp"] = community["camp"]
            graph.nodes[node_id]["opinion"] = float(np.clip(rng.normal(community["mean_opinion"], 0.09), -1.0, 1.0))
            graph.nodes[node_id]["enclave"] = int(community["enclave"])
            graph.nodes[node_id]["activity"] = int(rng.poisson(9 if community["enclave"] else 6) + 1)
            node_id += 1

    return graph


def write_demo_graph_files(seed=42):
    DATA_RAW.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    graph = build_demo_graph(seed=seed)

    node_frame = pd.DataFrame(
        [{"node_id": node, **attrs} for node, attrs in graph.nodes(data=True)]
    )
    edge_frame = pd.DataFrame(
        [{"source": source, "target": target} for source, target in graph.edges()]
    )

    nx.write_graphml(graph, DATA_RAW / "workshop_network.graphml")
    nx.write_gexf(graph, DATA_RAW / "workshop_network.gexf")
    nx.write_edgelist(graph, DATA_RAW / "workshop_network.edgelist", data=False)
    node_frame.to_csv(DATA_RAW / "workshop_nodes.csv", index=False)
    edge_frame.to_csv(DATA_PROCESSED / "workshop_edges.csv", index=False)


def load_graph(path):
    path = Path(path)
    if path.suffix == ".graphml":
        graph = nx.read_graphml(path)
    elif path.suffix == ".gexf":
        graph = nx.read_gexf(path)
    else:
        graph = nx.read_edgelist(path, nodetype=int)
        node_frame = pd.read_csv(DATA_RAW / "workshop_nodes.csv")
        attributes = node_frame.set_index("node_id").to_dict(orient="index")
        nx.set_node_attributes(graph, attributes)

    graph = nx.convert_node_labels_to_integers(graph, label_attribute="original_id")
    for _, attrs in graph.nodes(data=True):
        attrs["opinion"] = float(attrs["opinion"])
        attrs["activity"] = int(attrs["activity"])
        attrs["enclave"] = int(attrs["enclave"])
    return graph


In [ ]:
write_demo_graph_files()
G = load_graph(DATA_RAW / "workshop_network.graphml")


In [ ]:
def initialize_opinions(graph):
    return {node: float(graph.nodes[node]["opinion"]) for node in graph.nodes()}


def record_history(step, opinions, sample_nodes, records):
    for node in sample_nodes:
        records.append({"step": step, "node_id": node, "opinion": opinions[node]})


def count_clusters(opinions, tolerance=0.08):
    values = sorted(opinions.values())
    if not values:
        return 0
    clusters = 1
    for previous, current in zip(values, values[1:]):
        if abs(current - previous) > tolerance:
            clusters += 1
    return clusters


def summarise_state(label, opinions):
    values = np.array(list(opinions.values()))
    return {
        "scenario": label,
        "mean": round(float(values.mean()), 4),
        "std": round(float(values.std()), 4),
        "clusters": count_clusters(opinions),
        "range": round(float(values.max() - values.min()), 4),
    }


In [ ]:
def run_deffuant(graph, epsilon, mu=0.35, steps=2500, seed=42, sample_every=25):
    rng = np.random.default_rng(seed)
    edges = list(graph.edges())
    opinions = initialize_opinions(graph)
    sample_nodes = sorted(rng.choice(list(graph.nodes()), size=min(12, graph.number_of_nodes()), replace=False).tolist())
    records = []
    record_history(0, opinions, sample_nodes, records)

    for step in range(1, steps + 1):
        source, target = edges[rng.integers(len(edges))]
        if abs(opinions[source] - opinions[target]) <= epsilon:
            delta = mu * (opinions[target] - opinions[source])
            opinions[source] += delta
            opinions[target] -= delta
        if step % sample_every == 0:
            record_history(step, opinions, sample_nodes, records)

    return pd.DataFrame(records), opinions


def run_biased_exposure(graph, epsilon, bias_strength=6.0, mu=0.35, steps=2500, seed=42, sample_every=25):
    rng = np.random.default_rng(seed)
    opinions = initialize_opinions(graph)
    sample_nodes = sorted(rng.choice(list(graph.nodes()), size=min(12, graph.number_of_nodes()), replace=False).tolist())
    records = []
    record_history(0, opinions, sample_nodes, records)

    nodes = list(graph.nodes())
    for step in range(1, steps + 1):
        source = int(nodes[rng.integers(len(nodes))])
        neighbors = list(graph.neighbors(source))
        if neighbors:
            distances = np.array([abs(opinions[source] - opinions[target]) for target in neighbors])
            weights = np.exp(-bias_strength * distances)
            weights = weights / weights.sum()
            target = int(rng.choice(neighbors, p=weights))
            if abs(opinions[source] - opinions[target]) <= epsilon:
                delta = mu * (opinions[target] - opinions[source])
                opinions[source] += delta
                opinions[target] -= delta
        if step % sample_every == 0:
            record_history(step, opinions, sample_nodes, records)

    return pd.DataFrame(records), opinions


## 1. Phase transition under different confidence thresholds


In [ ]:
epsilons = [0.45, 0.25, 0.10]
dw_summaries = []
dw_histories = []

for epsilon in epsilons:
    history, final_state = run_deffuant(G, epsilon=epsilon, seed=42 + int(epsilon * 100))
    dw_histories.append(history.assign(model=f"DW epsilon={epsilon}"))
    dw_summaries.append(summarise_state(f"DW epsilon={epsilon}", final_state))

pd.DataFrame(dw_summaries)


In [ ]:
dw_history = pd.concat(dw_histories, ignore_index=True)
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=dw_history, x="step", y="opinion", hue="model", units="node_id", estimator=None, alpha=0.35, ax=ax)
ax.set_title("Opinion trajectories under different confidence thresholds")
plt.tight_layout()


## 2. Add algorithmic bias to the interaction rule


In [ ]:
unbiased_history, unbiased_state = run_deffuant(G, epsilon=0.25, seed=123)
biased_history, biased_state = run_biased_exposure(G, epsilon=0.25, bias_strength=7.5, seed=123)

pd.DataFrame(
    [
        summarise_state("Unbiased DW (epsilon=0.25)", unbiased_state),
        summarise_state("Biased neighbor selection", biased_state),
    ]
)


In [ ]:
trajectory_frame = pd.concat(
    [
        unbiased_history.assign(model="Unbiased"),
        biased_history.assign(model="Biased"),
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for axis, model_name in zip(axes, ["Unbiased", "Biased"]):
    chunk = trajectory_frame[trajectory_frame["model"] == model_name]
    sns.lineplot(data=chunk, x="step", y="opinion", units="node_id", estimator=None, alpha=0.4, ax=axis)
    axis.set_title(model_name)
plt.tight_layout()
